**Homework 13**

In this assignment we'll start working with PyTorch. We will recreate a small neural network, train a regression model on the cars data, and train an Iris classifier.

This is a completed notebook: run the cells from top to bottom. It uses the same manual gradient-descent updates as Homework 12, with PyTorch computing the derivatives. Google Colab already provides the packages used here; no GPU is needed.

The examples train and evaluate on the same data to focus on how PyTorch works. These are training metrics, not estimates of performance on unseen observations. A real evaluation needs held-out data.

In [ ]:
import torch
from torch.nn import Linear, ReLU, Sequential

torch.manual_seed(158)  # Reproducible initial weights and random examples

Let's use PyTorch to recreate the Neural Network from Homework 11:

In [ ]:
network=Sequential(
    Linear(2,3),
    ReLU(),
    Linear(3,1)
)

You should be able to use it the exact same way, except that we apply PyTorch models to PyTorch tensors, rather than Numpy arrays:

In [ ]:
X=torch.randn(15,2) #generate a random feature matrix with 2 features and 15 observations as a torch tensor
network(X) #Predictions of our network

You can see the weights and biases of your network as follows. Note that the first layer has a weight matrix of shape (3,2) and a bias vector of size 3, and the second layer has a weight matrix of shape(1,3) and a single bias.

In [ ]:
for param in network.parameters():
    print(param,param.shape)

We now use gradient descent to train our network. Let's create a target column with shape (15, 1), matching the predictions. For regression, both tensors must have the same shape so that each prediction is compared with its own target:

In [ ]:
target=torch.randn(15,1)

Our training loop now follows the pattern from Homework 12. The loss compares prediction with target, which we just defined.

The block beginning with `with torch.no_grad():` tells PyTorch not to record the parameter-update arithmetic in the differentiation graph. The next iteration builds a fresh graph for the new predictions.

In [ ]:
for i in range(10000): # Do 10000 gradient descent steps
  network.zero_grad() # Clear the derivative with respect to each parameter
  prediction=network(X)
  MSEloss=torch.nn.MSELoss()(prediction,target)
  # MSEloss=((prediction-target)**2).mean() # The same loss
  MSEloss.backward()

  with torch.no_grad():
    for param in network.parameters():
      param-=0.01*param.grad

  if i%1000==0:
    print(f"Step: {i}/10000, Loss: {MSEloss.item()}")

Let's now use this on real data. We'll use the same three columns from the cars dataset that we used in Homework 4, and again use the mpg column for our target.

The input features have very different scales. Standardize each column before training so that the learning rate works reasonably across all three features. We leave mpg in its original units.

The final layer produces shape (N, 1), so reshape the target to (N, 1) too. A flat target of shape (N,) would broadcast against (N, 1), creating N × N differences instead of one difference per car. If you later evaluate on held-out cars, use the means and standard deviations from the training data rather than recomputing them on the held-out data.

In [ ]:
import pandas as pd
cars=pd.read_csv('https://vincentarelbundock.github.io/Rdatasets/csv/causaldata/auto.csv')

DWG=torch.tensor(cars[['displacement','weight','gear_ratio']].to_numpy(),dtype=torch.float32)
DWG_mean=DWG.mean(dim=0)
DWG_std=DWG.std(dim=0)
DWG=(DWG-DWG_mean)/DWG_std
mpg=torch.tensor(cars.mpg.to_numpy(),dtype=torch.float32).reshape(-1,1)
print('Feature shape:', tuple(DWG.shape), 'Target shape:', tuple(mpg.shape))

The following network predicts mpg from the three standardized features in DWG. It has two hidden layers with 8 neurons and 4 neurons, each followed by ReLU. The final one-neuron layer has no activation because its output is a regression prediction:

In [ ]:
network=Sequential(
    Linear(3,8),
    ReLU(),
    Linear(8,4),
    ReLU(),
    Linear(4,1)
)

Train the network to predict mpg from DWG. Do 10000 gradient-descent steps with a learning rate of 0.001, and report the MSE every 1000 steps. The assertion checks that predictions and targets stay paired by row:

In [ ]:
for i in range(10000):
  network.zero_grad()
  prediction=network(DWG)
  assert prediction.shape==mpg.shape
  MSEloss=torch.nn.MSELoss()(prediction,mpg)
  MSEloss.backward()

  with torch.no_grad():
    for param in network.parameters():
      param-=0.001*param.grad

  if i%1000==0:
    print(f"Step: {i}/10000, Loss: {MSEloss.item()}")

Compute the final training MSE for the model, using the newly learned weights:

In [ ]:
with torch.no_grad():
  final_mse=torch.nn.MSELoss()(network(DWG),mpg).item()
final_mse

The value above is computed from your trained model; it is not a supplied answer to copy. It may vary slightly across PyTorch versions. It measures fit to the training cars, not generalization.

For classification problems, here are the changes:

1. If you are predicting n classes, the final layer should have n neurons and return raw scores (logits). Do not apply softmax before CrossEntropyLoss; that loss handles the normalization.
2. Use torch.nn.CrossEntropyLoss instead of MSE. The target has shape (N,) and contains integer class indices, so no one-hot encoding is needed. This is different from the regression target shape above.
3. To get predicted classes, use `torch.argmax(network(X), dim=1)`.
4. To see class probabilities, use `torch.softmax(network(X), dim=1)`.

With these changes in mind, we'll revisit the Iris dataset:

In [ ]:
from sklearn.datasets import load_iris
iris=load_iris()

X=torch.tensor(iris.data, dtype=torch.float32)
y=torch.tensor(iris.target, dtype=torch.long)

The classifier below maps the four Iris features to three class scores. It has two hidden layers, each with 10 neurons and followed by ReLU. Its output layer has no ReLU or softmax:

In [ ]:
iris_net=Sequential(
    Linear(4,10),
    ReLU(),
    Linear(10,10),
    ReLU(),
    Linear(10,3)
)

Train the classifier for 10000 steps with a learning rate of 0.01. Cross-entropy receives raw scores of shape (N, 3) and class indices of shape (N,):

In [ ]:
for i in range(10000):
  iris_net.zero_grad()
  pred=iris_net(X)
  CEloss=torch.nn.CrossEntropyLoss()(pred,y)
  CEloss.backward()

  with torch.no_grad():
    for param in iris_net.parameters():
      param-=0.01*param.grad

  if i%1000==0:
    print(f"Step: {i}/10000, Loss: {CEloss.item()}")

What probabilities does the trained model assign to the flower at row index 133 belonging to class index 1 or class index 2? All indices are zero-based:

In [ ]:
with torch.no_grad():
  probabilities=torch.softmax(iris_net(X),dim=1)
  class1_prob=probabilities[133,1].item()
  class2_prob=probabilities[133,2].item()
class1_prob,class2_prob

These two probabilities come from the current model. They need not sum to 1 by themselves because class index 0 also has a probability; all three entries in a row sum to 1.

Create a vector of predictions for your model:

In [ ]:
with torch.no_grad():
  predictions=torch.argmax(iris_net(X),dim=1)
predictions

Compute the training accuracy of your model:

In [ ]:
accuracy=(predictions==y).float().mean().item()
accuracy

Accuracy is the fraction of these training flowers whose predicted class equals the target. To measure performance on unseen flowers, train on one split and evaluate on a separate split. The notebook lab checks fixed shapes and operations rather than requiring a particular trained accuracy.